# CENG463 PA3

In this programming assignment, you will be dealing with encoder-based and decoder-based language models. You will use Python for this task. You can use libraries for your implementations, or implement your own functions. However, you are expected to analyse and reason about your implementation and results.

### IMPORTANT NOTES

* **Do not clear the output of your cells since this notebook will count as your written report and your cell outputs will be used for grading.** If a question in your submitted notebook does not have a printed output, you will get no grade from that question. If you encounter a problem about this, please [email me](mailto:auozturk@ceng.metu.edu.tr) so that we can work out a solution.

* Ideally, you should be able to complete this assignment on Google Colab, without any payment for resources to any service (or you can use your own computers). However, if you believe you need additional resources, please [send an email to Çağrı Hoca](mailto:ctoraman@metu.edu.tr).

## Problem: Misinformation Detection

On your last programming assignment, you will experiment with encoder-based and decoder-based models (specifically, BERT and Llama) to identify misinformation on English and Turkish tweet data from 2022. You can read the [paper](https://arxiv.org/pdf/2210.05401v2) for more information about the dataset. However, the dataset that is shared with you with the assignment is slightly simplified, where each row only consists of a `label` and `text` field.

A tweet is considered as "misinformation" if the `label` is `False`. Your models should aim to classify a given text according to the labels `True`, `False`, and `Other`.

**Put the "data" folder in the same directory with your notebook to work on your solutions in case we need to run your notebooks to reproduce your outputs.** The dataset consists of 5 train and test folds. For each model, you will train on 5 folds and report the average performance of all folds.

On the last part of the assignment, you will compare the time and space efficiency of the models you have used. So, don't forget to keep track of the training time.

**Important Note:** For this assignment, using tutorials, online forums, or AI tools to help with debugging and implementation may be needed. However, you are expected to add your resources as disclaimers. **You will be penalized if you do not disclose any help from AI tools, GitHub repositories, or tutorials.**

Tutorials/resources that might come handy:
* [`MiDe22`repository](https://github.com/metunlp/mide22)
* [`transformers` library](https://huggingface.co/docs/transformers/index)
* [finetuning a pretrained model](https://huggingface.co/docs/transformers/training)
* [`pipeline` from HuggingFace for inference](https://huggingface.co/docs/transformers/pipeline_tutorial##text-pipeline)
* [finetuning Llama](https://www.datacamp.com/tutorial/fine-tuning-llama-3-1)
* [quantization for working on low resources](https://huggingface.co/docs/transformers/v5.0.0rc0/quantization/overview)
* Or any other tutorial you find easy to follow. If you find a particularly helpful resource, you can share it in the discussion forum for everyone to use.

## Q1 - Encoder-based language models for classification (25 points)

### Part A: BERT and preprocessing

In this part, you will finetune the `google-bert/bert-base-uncased` model for the misinformation detection task for English tweets ("EN" folder in the dataset). However, you will train two versions: one with preprocessed texts, one with the raw texts.

For the preprocessing steps, lowercase the text, and use `nltk` to lemmatize and remove stopwords. You can also split each tweet into sentences and tokens, then combine the token list into a single space seperated string to represent each tweet as a single text again. Be careful not to combine different tweets. You can add additional preprocessing steps as long as you keep the integrity of the label-text mappings for each tweet.

You can also use the `preprocess_review` function from PA2 and then combine the returned list into a single space separated string.

Report the performance of both classifiers as the average of 5 train/test folds in "EN" dataset and accuracy, precision, recall, and F1-score with respect to the `False` class. Discuss which model performed better and why.

### Part B: BERT, Mutlilingual BERT and crosslingual performance

In this part, you will finetune the `google-bert/bert-base-uncased` and `google-bert/bert-base-multilingual-uncased` models and compare their performances. However, for the train/test folds, you will take the train fold from "EN" folder and the test fold with the corresponding number from the "TR" folder. This means that you will train the models on English but test them on Turkish.

Report the performance of both classifiers as the average of 5 train/test folds on accuracy, precision, recall, and F1-score with respect to the `False` class. Discuss which model performed better and why you think that is the case.



In [ ]:
# IMPLEMENT Q1 PART A HERE.
# YOU CAN ADD CELLS BELOW.

In [20]:
# Read the dataset from tsv file
# need to read each test and train tsv for EN and TR folders

import os
import pandas as pd

root = "data"

datasets = {
    "EN": {"train": {}, "test": {}},
    "TR": {"train": {}, "test": {}}
}

language_dict = {
    "EN": "english",
    "TR": "turkish"
}

for lang in language_dict.keys(): # ["EN", "TR"]
    lang_path = os.path.join(root, lang)
    

    #indexes for datasets
    train_idx = 0
    test_idx = 0
    
    for file in sorted(os.listdir(lang_path)):   # sorted = stable order
        if file.endswith(".tsv"):
            df = pd.read_csv(os.path.join(lang_path, file), sep="\t")

            # assign numeric order
            if "train" in file.lower():
                datasets[lang]["train"][train_idx] = df
                train_idx += 1
            elif "test" in file.lower():
                datasets[lang]["test"][test_idx] = df
                test_idx += 1

# example access
#print(datasets["EN"]["train"][1].head())
print(datasets["TR"]["train"][4].head())



   label                                               text
0  False  Izmir mv @avhamzadag neden izmirde multecilere...
1  Other  Açız, mutfak masrafları, Kira, okul giderleri,...
2  False  @yyunikonn İzmire yığılma olur diye yorumladim...
3   True  ❌ MEB'in İzmir’de mültecilere özel dört okul y...
4  Other  @wolfintoaction Çok gerekliyse ülkeye göçmen d...


In [23]:
## Preprocessing Steps
import re

# taken from PA1 and apoted for our needs
def remove_usernames(text):
    # valid twitter usernames = @ + 2–15 chars alnum or underscore
    return re.sub(r'@([A-Za-z0-9_]{2,15})', '', text)

def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Symbols & pictographs
    "\U0001F680-\U0001F6FF"  # Transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # Flags
    "\U00002700-\U000027BF"  # Dingbats
    "\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs
    "]+",
    flags=re.UNICODE
)

def remove_emojis(text):
    return emoji_pattern.sub('', text)


# this regex taken from AI
def remove_numbers(text):
    # remove standalone numeric-like tokens
    return re.sub(r'\b[\d.,:]+\b', '', text)

def replace_links(text): #replace links with [URL]
    return re.sub(r'http\S+|www\S+|https\S+', '[URL]', text, flags=re.MULTILINE)

# Taken from PA2 and modified for our needs
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')  
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review(text, lang='english'):
    # remove unwanted patterns
    text = replace_links(text) # i add this since in datasets there are too many links
    text = remove_usernames(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = remove_numbers(text)

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    stop_words = set(stopwords.words(lang))
    lemmatizer = WordNetLemmatizer()
    
    sentences = sent_tokenize(text, language=lang) # it takes language parameter, we did not use it before since we only had english reviews before 
    
    result_tokens = []
    
    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]
        
        result_tokens.extend(lemmatized_sentence)
    
    return " ".join(result_tokens)

[nltk_data] Downloading package wordnet to /Users/ovak/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/ovak/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/ovak/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [24]:
# test the preprocessing function
sample_text = "@dpa_live @dpa @onetz_de @_FriedrichMerz @n_roettgen @HBraun @EschleThomas @CDU @CSU @Markus_Soeder @derspiegel @welt @tagesschau @ZDF @SZ @Tagesspiegel @BR24 @maybritillner @AnneWillTalk @heutejournal @cnnbrk @faznet @washingtonpost @nytimes @OlafScholz @ABaerbock @c_lindner @NancyFaeser @MarcoBuschmann @ScottMorrisonMP @karenandrewsmp @ashbarty @AustralianOpen @DjokerNole @BILD @EP_President @DavidSassoli @AlexHawkeMP @PeterDutton_MP @usopen @Olympics @FIFAcom @FIFAWorldCup @Pasferre @francefootball @Cristiano @TomBrady @mosseri @instagram @POTUS #UKRAINE PRES.REFUSED #PEACE INITIATED BY #SCHOLZ FEW DAYS BEFORE #RUSSIA ATTACKED=U RESPONSIBLE f.MANY DEATH+WAR TOO+#EINSTEIN:WORLD BACK #STONEAGE IF WW3+#POLAND SILLY WANT USA #NUCLEAR+GER MOBBERS MULTIPLE MORE SADISTIC TO ME=NO #WAR END YOUR OWN #SINS https://t.co/HYSfo3Nsfg."
preprocessed_text = preprocess_review(sample_text, lang='english')
print("Original Text:", sample_text)
print("Preprocessed Text:", preprocessed_text)

Original Text: @dpa_live @dpa @onetz_de @_FriedrichMerz @n_roettgen @HBraun @EschleThomas @CDU @CSU @Markus_Soeder @derspiegel @welt @tagesschau @ZDF @SZ @Tagesspiegel @BR24 @maybritillner @AnneWillTalk @heutejournal @cnnbrk @faznet @washingtonpost @nytimes @OlafScholz @ABaerbock @c_lindner @NancyFaeser @MarcoBuschmann @ScottMorrisonMP @karenandrewsmp @ashbarty @AustralianOpen @DjokerNole @BILD @EP_President @DavidSassoli @AlexHawkeMP @PeterDutton_MP @usopen @Olympics @FIFAcom @FIFAWorldCup @Pasferre @francefootball @Cristiano @TomBrady @mosseri @instagram @POTUS #UKRAINE PRES.REFUSED #PEACE INITIATED BY #SCHOLZ FEW DAYS BEFORE #RUSSIA ATTACKED=U RESPONSIBLE f.MANY DEATH+WAR TOO+#EINSTEIN:WORLD BACK #STONEAGE IF WW3+#POLAND SILLY WANT USA #NUCLEAR+GER MOBBERS MULTIPLE MORE SADISTIC TO ME=NO #WAR END YOUR OWN #SINS https://t.co/HYSfo3Nsfg.
Preprocessed Text: presrefused initiated day attacked=u responsible fmany death+war too+ : world back ww3+ silly want usa +ger mobbers multiple sadis

In [25]:
#turkish test
sample_text_tr = "Hangi akla hizmetse" + """
Karşıyaka Bostanlı da
Kiraların 3.000 TL olduğu
1 tane Suriyeli nin yaşamadığı
Bostanlının göbeğinde
Yıllardır okul yapılmayan arsaya
Suriyeliler için
Mülteci Okulu yapıyorlar

Okulumu yıktılar https://t.co/cGflKdPzLa #gazetesozcu @gazetesozcu"""
preprocessed_text = preprocess_review(sample_text_tr, lang='turkish')
print("Original Text: \n", sample_text_tr)
print("-"*10)
print("Preprocessed Text:\n", preprocessed_text)

Original Text: 
 Hangi akla hizmetse
Karşıyaka Bostanlı da
Kiraların 3.000 TL olduğu
1 tane Suriyeli nin yaşamadığı
Bostanlının göbeğinde
Yıllardır okul yapılmayan arsaya
Suriyeliler için
Mülteci Okulu yapıyorlar

Okulumu yıktılar https://t.co/cGflKdPzLa #gazetesozcu @gazetesozcu
----------
Preprocessed Text:
 hangi akla hizmetse karşıyaka bostanlı kiraların tl olduğu tane suriyeli nin yaşamadığı bostanlının göbeğinde yıllardır okul yapılmayan arsaya suriyeliler mülteci okulu yapıyorlar okulumu yıktılar [ url ]


In [ ]:
# apply preprocessing to all datasets with new column prep_text
for lang in datasets.keys(): # ["EN", "TR"]
    lang_name = language_dict[lang]
    for split in datasets[lang].keys(): # ["train", "test"]
        for idx in datasets[lang][split].keys():
            df = datasets[lang][split][idx]
            df['prep_text'] = df['text'].apply(lambda x: preprocess_review(x, lang=lang_name))
            datasets[lang][split][idx] = df


   label                                               text  \
0  Other  @benjaminhaddad Now maybe you can rethink the ...   
1  Other  Poland, Hungary and now Ukraine via Zelensky a...   
2  Other  @therrienv @stupidmaggats @417craig @Yeshua17F...   
3  Other  @dpa_live @dpa @onetz_de @_FriedrichMerz @n_ro...   
4  Other  @Podolyak_M Americans stand with the people of...   

                                           prep_text  
0  maybe rethink train trip boasting `` come back...  
1  poland , hungary ukraine via zelensky saying ’...  
2  decision biden make . zelensky said need , pol...  
3  presrefused initiated day attacked=u responsib...  
4                            american stand people !  


In [27]:
# check db
print(datasets["EN"]["train"][0].head())
print("*"*10)
print(datasets["TR"]["train"][0].head())

   label                                               text  \
0  Other  @benjaminhaddad Now maybe you can rethink the ...   
1  Other  Poland, Hungary and now Ukraine via Zelensky a...   
2  Other  @therrienv @stupidmaggats @417craig @Yeshua17F...   
3  Other  @dpa_live @dpa @onetz_de @_FriedrichMerz @n_ro...   
4  Other  @Podolyak_M Americans stand with the people of...   

                                           prep_text  
0  maybe rethink train trip boasting `` come back...  
1  poland , hungary ukraine via zelensky saying ’...  
2  decision biden make . zelensky said need , pol...  
3  presrefused initiated day attacked=u responsib...  
4                            american stand people !  
**********
   label                                               text  \
0   True  MEB'den İzmir'deki okul için açıklama: Ağaçlar...   
1  False  @KORDELYAz okul yapılıyor ama bir fark var. mü...   
2   True  'Mültecilere okul' yalanıyla kışkırtma peşinde...   
3   True  **İstiklal madalya

**DISCUSS Q1 PART A HERE.**

Q1-A discussion.

In [ ]:
# IMPLEMENT Q1 PART B HERE.
# YOU CAN ADD CELLS BELOW.

In [ ]:
# IMPLEMENT Q1 PART B

**DISCUSS Q1 PART B HERE.**

Q1-B disscusion.

## Q2 - Decoder-based language models for classification (55 points)

### Part A: Zero-shot and Few-shot inference with Llama for text classification

In this part, you will use the `meta-llama/Llama-3.1-8B-Instruct` model and the "EN" English tweets dataset for designing a classifier based on inference. You will design two prompts:

* **Prompt 1:** A zero-shot prompt with only the task description, no examples. Explain the task and ask for a classification of the tweets in test folds.
* **Prompt 2:** A one-shot or few-shot prompt, with one or a couple examples. This can be tricky as longer tweets may cause problems. Explain the task using examples from the training folds, and ask for a classification of the tweets in test folds.

Report the performance of both classification pipelines as the average of 5 test folds on accuracy, precision, recall, and F1-score with respect to the `False` class. Explain your prompt design process in detail and discuss the performance of the models with respect to your prompts.

### Part B: Finetuning Llama

In this part, you will again use `meta-llama/Llama-3.1-8B-Instruct` model and the "EN" English tweets dataset but this time you will try to finetune the model with the training folds. However, this may become tricky and resource demanding, considering you are using free tools (such as Google Colab) or your own computers. Check out the tutorials to see how you can better manage your resources.

Report the performance of the finetuned classifier as the average of 5 test folds on accuracy, precision, recall, and F1-score with respect to the `False` class, with the better performing prompt you have designed in Part A. Discuss the effect of finetuning on the model performance. Do you think it improved the performance? What else can be integrated into the process to improve the performance of the model, specifically for the misinformation detection task?


### Part C: Domain transfer to sentiment analysis

In this part, you will try to classify game reviews with the model you finetuned in Part B to see the effect of finetuning on the generalized performance of the model.

You should design a simple zero-shot prompt that asks the model whether a user recommends the game given their review text. With the same prompt, you will use both the base `meta-llama/Llama-3.1-8B-Instruct` model and your finetuned version to classify the reviews.

Report the performance of both classifiers on the whole dataset with respect to accuracy, precision, recall, and F1-score. Discuss and explain your findings.

**Note: Dataset for Part C**

* You will use the `game_reviews.csv` file shared with you. It is taken from [this Kaggle dataset](https://www.kaggle.com/datasets/piyushagni5/sentiment-analysis-for-steam-reviews). Review text attribute is `user_review`. User recommendation is the attribute `user_suggestion` where `1` means user recommended the game, `0` means user did not recommend the game.

In [ ]:
# IMPLEMENT Q2 PART A HERE.
# YOU CAN ADD CELLS BELOW.

**DISCUSS Q2 PART A HERE.**

In [ ]:
# IMPLEMENT Q2 PART B HERE.
# YOU CAN ADD CELLS BELOW.

**DISCUSS Q2 PART B HERE.**

In [ ]:
# IMPLEMENT Q2 PART C HERE.
# YOU CAN ADD CELLS BELOW.

**DISCUSS Q2 PART C HERE.**

## Q3 - Discussion of classification performance and resource use (20 points)

In this part, you are expected to discuss the misinformation detection performance and resource use of the models you have trained in Q1 Part A-B and Q2 Part A-B. Which model performs better? Which model uses more resources? Please discuss your OWN findings with YOUR OWN WORDS. Do not forget to share the time and resource need details you have measured and observed.

**WRITE YOUR DISCUSSION FOR Q3 HERE.**